# 06 Comparative analysis (A vs B vs C)

**Question.** How do B and C differ from A on the validation partition, with uncertainty?

**Type.** Formal comparison on validation predictions via paired bootstrap (resampling unit: image; optimistic because patient grouping is unknown). Uses the latest completed run per experiment; set RUN IDs explicitly for a final comparison.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
from ct_iqa.experiment.runs import latest_completed_run, read_predictions
from ct_iqa.evaluation.bootstrap import paired_bootstrap_difference
ids = {"A": "A_reference_resnet50", "B": "B_ranking_aware_resnet50", "C": "C_uncertainty_aware_resnet50"}
runs = {k: latest_completed_run(v) for k, v in ids.items()}
print(runs)
preds = {k: read_predictions(r) for k, r in runs.items() if r}

In [ ]:
def compare(other):
    if "A" not in preds or other not in preds:
        print(f"{other} vs A: PENDING (missing completed run)"); return
    a, b = preds["A"], preds[other]
    assert a["filename"] == b["filename"], "runs must share the validation partition"
    res = paired_bootstrap_difference(a["label_0_4"], a["prediction_0_4"], b["prediction_0_4"], n_boot=2000, seed=0)
    for m, r in res.items():
        print(f"{other} - A  {m:>5}: {r['difference_b_minus_a']:+.4f}  95% CI [{r['ci_low']:+.4f}, {r['ci_high']:+.4f}]")
compare("B"); compare("C")

Interpretation rules: [EXPERIMENT_PROTOCOL](../docs/experiment-protocol.md). No variant is declared better when the interval includes zero. The official test set is not used here.